In [1]:
NUM_WORKERS_FOR_LOADER = 2

In [2]:
from typing import Literal

from torch.utils.data import DataLoader
import torch

from datasets import load_dataset, load_from_disk
from torch.utils.data import Dataset
import pandas as pd
from pandas import DataFrame
import os

from functools import partial

class YambdaDataset(Dataset):

    DEFAULT_PATH = './datasets/yambda_likes_dataset'
    SECONDS_IN_DAY = 24 * 60 * 60

    def __init__(self, path : str | None = None,
                 dataset_type : Literal['50m', '500m', '5b'] = '50m',
                 overwrite : bool = False,
                 mode : Literal['train', 'val'] = 'train',
                 max_seq_len : int = 256):

        self.path = path if path is not None else YambdaDataset.DEFAULT_PATH
        self.path += dataset_type

        self.dataset = None

        if overwrite or not os.path.exists(self.path):
            self.dataset = load_dataset("yandex/yambda", data_dir=f"flat/{dataset_type}", data_files="likes.parquet")
            self.dataset.save_to_disk(self.path)
        else:
            self.dataset = load_from_disk(self.path)

        self.dataset = DataFrame(self.dataset['train'])

        self.dataset['item_id'], _ = pd.factorize(self.dataset['item_id'])

        self.pad_id = self.dataset['item_id'].max() + 1

        start = self.dataset['timestamp'].min()
        end = self.dataset['timestamp'].max()

        if mode == 'train':
            self.dataset = self.dataset[self.dataset['timestamp'] < end - 7 * self.SECONDS_IN_DAY]
        else:
            self.dataset = self.dataset[self.dataset['timestamp'] >= end - 7 * self.SECONDS_IN_DAY]

        self.dataset : DataFrame
        self.dataset.sort_values(by=['timestamp'], inplace=True)
        self.dataset['num'] = self.dataset.groupby('uid').cumcount() // max_seq_len
        self.dataset = self.dataset.groupby(['uid', 'num'])['item_id'].apply(list).reset_index()

    def __getitem__(self, idx) -> torch.Tensor:
        return torch.tensor(self.dataset.iloc[idx]['item_id'], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.dataset)


class Utils:

    @staticmethod
    def collate_to_batch(batch, pad_id, max_len):
        batch_t = torch.tensor([[seq[i] if i < len(seq) else pad_id for i in range(max_len)] for seq in batch], dtype=torch.long)
        return batch_t

    @staticmethod
    def collate_with_random_negatives(batch, pad_id, num_neg, max_len):
        batch_t = Utils.collate_to_batch(batch, pad_id, max_len)
        neg = torch.randint(0, pad_id, (*batch_t.shape, num_neg), dtype=torch.long)
        return [batch_t, neg]

    @staticmethod
    def get_train_dataloader(batch_size=32, max_len=200, num_neg=256,
                            dataset_type: Literal['50m', '500m', '5b'] = '500m'):
        dataset = YambdaDataset(max_seq_len=max_len, mode='train', dataset_type=dataset_type)

        collate_fn = partial(
            Utils.collate_with_random_negatives,
            pad_id=dataset.pad_id,
            num_neg=num_neg,
            max_len=max_len
        )

        dataloader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True,
                                num_workers=NUM_WORKERS_FOR_LOADER,
                                collate_fn=collate_fn)
        return dataloader

    @staticmethod
    def get_val_dataloader(batch_size=32, max_len=200,
                        dataset_type: Literal['50m', '500m', '5b'] = '500m'):
        dataset = YambdaDataset(max_seq_len=max_len, mode='val', dataset_type=dataset_type)

        collate_fn = partial(
            Utils.collate_to_batch,
            pad_id=dataset.pad_id,
            max_len=max_len
        )

        dataloader = DataLoader(dataset=dataset, batch_size=batch_size,
                                num_workers=NUM_WORKERS_FOR_LOADER,
                                collate_fn=collate_fn)
        return dataloader



In [3]:
from torch import nn
import torch


class SASRec(nn.Module):
    def __init__(self, num_embedding: int,
                 seq_len: int,
                 embedding_dim: int = 256,
                 num_heads: int = 4,
                 num_layers: int = 4,
                 transformer_dim = 2048,
                 transformer_dropout = 0.1,
                 reuse_embeddings: bool = False):
        super(SASRec, self).__init__()
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.num_layers = num_layers

        self.input_embedding = nn.Embedding(num_embeddings=num_embedding + 1, embedding_dim=embedding_dim)
        self.position_embedding = nn.Embedding(num_embeddings=seq_len, embedding_dim=embedding_dim)

        self.transformer = nn.TransformerEncoder(
            encoder_layer=nn.TransformerEncoderLayer(
                d_model=embedding_dim,
                nhead=num_heads,
                dim_feedforward=transformer_dim,
                dropout=transformer_dropout,
                batch_first=True
            ),
            num_layers=num_layers
        )

        self.linear = nn.Linear(in_features=embedding_dim, out_features=embedding_dim)

        if reuse_embeddings:
            self.output_embedding = self.input_embedding
        else:
            self.output_embedding = nn.Embedding(num_embeddings=num_embedding + 1, embedding_dim=embedding_dim)

        self.pad_id = num_embedding

    def forward(self, x):

        device = x.device

        input_embeddings = self.input_embedding(x)

        seq_len = input_embeddings.shape[1]

        positions = (torch.arange(seq_len, dtype=torch.long, device=device)
                     .unsqueeze(0).expand(x.size(0), seq_len))
        position_embeddings = self.position_embedding(positions)

        embeddings = input_embeddings + position_embeddings

        mask = (x == self.pad_id)

        casual_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(device)

        attention = self.transformer(embeddings, mask=casual_mask,
                                     src_key_padding_mask=mask)

        return self.linear(attention)

In [4]:
import torch
from torch.nn.functional import binary_cross_entropy_with_logits, cross_entropy

import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

def show_metrics(metric1, metric2, label1, label2, name):
    plt.figure(num=name)
    plt.plot(range(len(metric1)), metric1, label=label1)
    plt.plot(range(len(metric2)), metric2, label=label2)
    plt.legend()
    plt.grid(True)
    plt.show()

def train_epoch(model : SASRec, train_loader, optimizer):
    model.train()
    sum_loss = 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for i, (positives, negatives) in enumerate(tqdm(train_loader, desc="Batch")):
        optimizer.zero_grad()
        positives, negatives = positives.to(device), negatives.to(device)

        model_input = positives[:, :-1]  # B, S, E
        positives = positives[:, 1:]
        negatives = negatives[:, 1:, :] # B, S, N, E
        neg_embeddings = model.output_embedding(negatives)
        pos_embeddings = model.output_embedding(positives)

        output = model(model_input)
        neg_logits = torch.einsum("bse, bsne -> bsn", output, neg_embeddings)
        pos_logits = torch.einsum("bse, bse -> bs", output, pos_embeddings)

        pos_logits = pos_logits.unsqueeze(-1)

        logits = torch.cat([neg_logits, pos_logits], dim=-1)
        gt = torch.cat([torch.zeros_like(neg_logits), torch.ones_like(pos_logits)], dim=-1)

        loss = binary_cross_entropy_with_logits(logits, gt, reduction="none")

        mask = (positives != model.pad_id)
        mask = mask.unsqueeze(-1).float()

        loss = loss * mask
        loss = loss.sum() / mask.sum()

        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    return sum_loss / len(train_loader)

@torch.no_grad()
def validate_epoch(model: SASRec, val_loader):
    model.eval()

    sum_loss = 0
    correct_top1 = 0
    correct_top5 = 0
    total = 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    all_embeddings = model.output_embedding.weight  # V, E

    for labels in tqdm(val_loader, desc="Batch"):
        labels = labels.to(device)

        model_input = labels[:, :-1]
        labels = labels[:, 1:]

        model_output = model(model_input)  # B, S, E
        logits = torch.einsum("bse, ve -> bsv", model_output, all_embeddings)

        logits = logits.reshape(-1, logits.size(-1))
        labels = labels.reshape(-1)

        loss = cross_entropy(logits, labels)
        sum_loss += loss.item()

        mask = (labels != model.pad_id).float()
        total += mask.sum().item()

        top1 = logits.argmax(dim=-1)
        top5 = logits.topk(5, dim=-1).indices

        correct_top1 += ((top1 == labels) * mask).sum().item()
        correct_top5 += (((top5 == labels.unsqueeze(-1)).any(dim=-1)) * mask).sum().item()

    avg_loss = sum_loss / len(val_loader)
    top1_acc = correct_top1 / total
    top5_acc = correct_top5 / total

    return avg_loss, top1_acc, top5_acc



def train(model : SASRec, train_loader, val_loader, optimizer, epochs):
    train_losses, val_losses = [], []
    top1_accs = []
    top5_accs = []

    for epoch in tqdm(range(epochs)):
        sum_loss, top1_acc, top5_acc = validate_epoch(model, val_loader)

        val_losses.append(sum_loss)
        top1_accs.append(top1_acc)
        top5_accs.append(top5_acc)

        train_losses.append(train_epoch(model, train_loader, optimizer))

        print(f"Epoch {epoch}: train_loss={train_losses[-1]:.4f}, val_loss={val_losses[-1]:.4f}, "
              f"top1={top1_acc:.4f}, top5={top5_acc:.4f}")

        show_metrics(train_losses, val_losses, "train", "val", "losses")
        show_metrics(top1_accs, top5_accs, "top-1", "top-5", "accuracy")


In [5]:
dataset_type = '500m'

In [6]:
train_loader = Utils.get_train_dataloader(dataset_type=dataset_type)
val_loader = Utils.get_val_dataloader(dataset_type=dataset_type)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

flat/500m/likes.parquet:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9033960 [00:00<?, ? examples/s]

In [7]:
dataset = YambdaDataset(dataset_type=dataset_type)

In [8]:
model = SASRec(num_embedding=dataset.pad_id + 1, seq_len=200)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
device

device(type='cuda')

In [11]:
optimizer = torch.optim.Adam(model.parameters())

train(model, train_loader, val_loader, optimizer, 20)

  0%|          | 0/20 [00:00<?, ?it/s]

Batch:   0%|          | 0/1240 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 15.14 GiB. GPU 0 has a total capacity of 14.74 GiB of which 13.31 GiB is free. Process 4953 has 1.43 GiB memory in use. Of the allocated memory 1.26 GiB is allocated by PyTorch, and 43.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [12]:
dataset.pad_id

638230